# 1. Simulated Scenario
I'm an Analyst on the Customer Experience team at a Brazilian online marketplace (think: Olist, ~2018). In a leadership sync, the VP of Ops says:
"Our review scores have been softer lately, and we're getting more delivery-related complaints. Before we make any big calls on logistics spend, can you dig into what's actually going on and tell us what we should do?"

# 2. Clarifying Questions I'd Ask/Think Through
To find out what's actually going on and what we should do, I would first break down the problem to identify the goal or the problem we're trying to address. After defining the goal, it is also important to communicate the priority of this problem, and the actual resource (data) we have in hand. This process is important as it assures that everyone's on the same page and keep the team's work/goal consistent.
## Define The Problem  
The review scores are softer compared to what/when?   
 - Is it a sudden drop (could imply system errors or change in platform interface/policies/carriers...etc.) or a trend?
 - Is the drop in a certain group/segment/region?   
What defines a lower review score?   
 - Are we measuring by the mean, mode, or other?    
 - Are there any outliers?    
Should I be testing the assumption that delivery-related complaints are causing lower review scores?  
 - What counts as delivery-related complaints?   
 - Covariates?
## Practical Constraints
How urgent is this? 
 - How detail should i go into this problem? Should I prioritize this over other tasks I have in hand?
 - When Should the team gather and discuss again?
What decision would my analyses support? (e.g., adjusting logistics spend?)  
 - What actual options we have on the table? (e.g., change communication to customers? negotiate with carriers?)

These questions might not all have a clear/immediate answer, but thinking through it helps me map out the initial goal and analysis plan.

# 3. Data Sanity Check
Since this is a simulated scenario, I'd like to look into the actual data before making any assumptions.
## Timeframe and Baseline Assumption

In [20]:
# Loading libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sys
from pathlib import Path

sys.path.append("../src")
from helpers import get_connection, sql
pd.set_option('display.max_colwidth', None)

In [ ]:
# Loading datasets as tables into SQLite
conn = get_connection(["orders", "reviews", "order_items", "customers"])

In [3]:
# Check the time span of the dataset
sql("" \
"SELECT MIN(order_purchase_timestamp) AS earliest_order, " \
"MAX(order_purchase_timestamp) AS latest_order " \
"FROM orders;", conn)

,earliest_order,latest_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [ ]:
# Check the mean of review scores by month
sql("" \
"SELECT strftime('%Y-%m', review_creation_date) AS year_month, " \
"ROUND(AVG(review_score), 2) AS avg_score, " \
"COUNT(*) AS num_reviews " \
"FROM reviews " \
"GROUP BY year_month " \
"ORDER BY year_month;", conn)

,year_month,avg_score,num_reviews
0,2016-10,4.06,179
1,2016-11,3.19,101
2,2016-12,2.36,45
3,2017-01,4.34,239
4,2017-02,4.28,1413
5,2017-03,4.03,2481
6,2017-04,4.04,2055
7,2017-05,4.10,3710
8,2017-06,4.13,3440
9,2017-07,4.18,3503


Given that the VP in the simulated scenario said "the review scores have been softer lately", I'm assuming this scenario happened in early April 2018, whereas "lately" refers to Jan-Mar 2018. For the baseline, I'm assuming the review scores are softer compared to 2017's average, as it's the most stable reference point here. However, it's possible that Jan-Mar has a recurring seasonal dip that is independent of other specific issue. I will come back to this in later analyses.

## Feasibility of review-related data
Below, I'm checking the richness of the review comments to make sure that it's feasibile if we ever want to inspect the reason of low reviews by looking into the comments.

In [23]:
sql("""
SELECT
    review_score,
    COUNT(*) AS total_reviews,
    COUNT(review_comment_message) AS with_comment
FROM reviews
GROUP BY review_score
ORDER BY review_score;
""", conn)

,review_score,total_reviews,with_comment
0,1,11424,8745
1,2,3151,2145
2,3,8179,3557
3,4,19142,5976
4,5,57328,20554


In [32]:
df = sql("""
SELECT
    review_comment_message,
    review_score
FROM reviews
WHERE review_score <= 3 AND review_comment_message IS NOT NULL
ORDER BY random()
LIMIT 10;
""", conn)

In [33]:
df.sample(n=5, random_state=51)

,review_comment_message,review_score
3,"O produto que eu pedi não foi entregue, entregaram outro no lugar,\r\nFalta de respeito da loja lannister com os seus clientes!",1
4,Até o momento não recebi o produto. Aguardando.,1
2,Nunca fiz uma compra pela lannister que demorasse tanto a receber. Gostaria de saber o motivo!,1
6,O produto não veio,1
7,Não recebi o produto e até o momento não obtivemos retorno em relação a devolução da fatura e pagamento,1


The amount and the content of the review comments both seem rich enough for related analyses. Other than that, I Google Translated the 5 sampled comments above, and found that the complaint issues could be classified as product quality (index 3, 4), delivery (index 2, 7), and order packing error (index 6). This suggests that issues other than delivery could also be contributing to the lower review score.

Additionaly, since the VP mentioned that we're getting more delivery-related complaints, another way to inspect that is to infer by the actual delivered date and the estimated delivery date, which lives in the orders table (quick gimpse to confirm below).

In [25]:
sql("" \
"SELECT * FROM orders " \
"LIMIT 5;", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


# 4. Hypothesis
The VP's framing implies delivery issues are driving the softening, but per my clarifying questions above, this is exactly the kind of claim worth testing rather than accepting. I'm treating it as a **working hypothesis** instead of a confirmed cause:

> *The Jan–Mar 2018 softening in review scores is at least partially attributable to delivery-related issues (e.g., late delivery vs. estimate, damaged goods).*

My analysis will test this, using both the actual-vs-estimated delivery date fields and comment text, and will check for alternative explanations (seasonality, pricing, product category shifts) before attributing the drop to delivery specifically.


# 5. Candidate Framings
Before committing fully to the delivery hypothesis, it's worth naming the other plausible ways to scope this investigation, the comment sample above already surfaced at least two alternatives:

**A. Delivery / logistics-driven** — the softening is primarily about delivery timing (late vs. estimate) or damaged goods. Testable using `order_estimated_delivery_date` vs. `order_delivered_customer_date`, which I confirmed exist and look usable.

**B. Product quality / listing accuracy-driven** — the softening is about products not matching their descriptions or being defective, independent of delivery. The sample comments (index 3, 4) support this as a real, distinct theme. Testable via product category patterns and comment classification.

**C. Segment-specific, not universal** — the softening isn't a general trend at all, but concentrated in a specific region, seller, or category, with a few bad actors dragging down the overall average. Testable via `GROUP BY` region/seller/category on review score. (If a regional pattern emerges, cross-referencing with external demographic data could help explain *why* — e.g., whether lower-income regions see worse delivery infrastructure, different price sensitivity, or different product mix. This is outside the Olist dataset itself and would only be worth pursuing if Framing C shows a real regional signal.)

**Prioritization:** I'm starting with Framing A, since it directly addresses the VP's stated concern and I've already confirmed the data supports testing it. If delivery doesn't fully explain the Jan–Mar dip, I'll follow up with B and C as secondary investigations.

# 6. Final Scoped Question

**Primary question:** Did delivery performance (actual vs. estimated delivery date) meaningfully decline in Jan–Mar 2018 compared to the 2017 baseline, and does that decline statistically explain a meaningful share of the drop in average review score over the same period?

**Secondary question, if needed:** If delivery performance doesn't fully explain the drop, do product-quality or fulfillment-related complaints (via comment classification) account for the remainder?

**Out of scope for this pass:** seller-level interventions, building a predictive model, or a full NLP sentiment pipeline. (this is a diagnostic pass to confirm the delivery hypothesis and identify what the VP should act on.)

#